# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 3: Data Cleaning, Transformation, and Feature Engineering

This revised Step 3 creates the analysis-ready layer used by Steps 4 through 9. It combines controlled cleaning with explicit transformations and domain-oriented feature engineering for complaint, temporal, geographic, victim, suspect, relationship, and violation analysis.

### Principles

1. Preserve `df_raw` unchanged.
2. Apply corrections to `df_clean` only.
3. Record transformations in an audit log.
4. Preserve Portuguese accents in display values.
5. Use normalized keys only for comparison and joins.
6. Do not remove sparse columns automatically.
7. Do not remove repeated `hash` values until the analytical grain is established.
8. Keep raw variables and derived features side by side.
9. Avoid target leakage and future information in model-oriented features.
10. Exclude numeric missing codes `-95`, `-97`, `-98`, `-99`, and `-999` from the research rules.


## 3.1 Imports and configuration


In [27]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

DATA_PATH = Path("data/disque100.csv")
SHEET_NAME = 0
OUTPUT_DIR = Path("outputs/step_03_data_cleaning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REMOVE_EXACT_DUPLICATES = False
EXPORT_CLEAN_DATA = True
EXPORT_FORMAT = "parquet"
CREATE_MODEL_MATRIX = False
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()


## 3.2 Load and fingerprint the source


In [28]:
def load_dataset(path, sheet_name=0):
    ext=path.suffix.lower()
    if ext==".csv":
        try: return pd.read_csv(path,sep=None,engine="python",encoding="utf-8",low_memory=False)
        except UnicodeDecodeError: return pd.read_csv(path,sep=None,engine="python",encoding="latin-1",low_memory=False)
    if ext in {".xlsx",".xls"}: return pd.read_excel(path,sheet_name=sheet_name)
    if ext==".parquet": return pd.read_parquet(path)
    raise ValueError("Supported formats: CSV, XLSX/XLS, Parquet")

def sha256_file(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

if DATA_PATH.exists():
    source_sha256=sha256_file(DATA_PATH)
    df_raw=load_dataset(DATA_PATH,SHEET_NAME)
    df_clean=df_raw.copy(deep=True)
    print(df_raw.shape, source_sha256)
else:
    df_raw=df_clean=None
    source_sha256=None
    print("Update DATA_PATH:",DATA_PATH.resolve())


Update DATA_PATH: C:\Projetos\eda-disque100-completo\eda\data\disque100.csv


## 3.3 Cleaning dictionaries and audit helpers


In [29]:
TEXT_MISSING_TOKENS = {
    "", "-", "--", ".", "..", "...", "N/A", "NA", "NAN", "NULL", "NONE",
    "NÃO SABE INFORMAR", "N/D", "NÃO INFORMADO", "NÃO SE APLICA",
    "DENUNCIANTE NÃO SOUBE INFORMAR", "NÃO SABE", "ATENDIMENTO INTERROMPIDO",
}

# Values that express a negative answer rather than missingness.
NEGATIVE_EQUIVALENTS = {
    "NÃO TEM DEFICIÊNCIA": "NÃO",
    "NÃO": "NÃO",
    "NÃO TEM DOENÇA RARA": "NÃO",
}

UF_CODES={"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}
REGION_BY_UF={
"AC":"NORTE","AP":"NORTE","AM":"NORTE","PA":"NORTE","RO":"NORTE","RR":"NORTE","TO":"NORTE",
"AL":"NORDESTE","BA":"NORDESTE","CE":"NORDESTE","MA":"NORDESTE","PB":"NORDESTE","PE":"NORDESTE","PI":"NORDESTE","RN":"NORDESTE","SE":"NORDESTE",
"DF":"CENTRO-OESTE","GO":"CENTRO-OESTE","MT":"CENTRO-OESTE","MS":"CENTRO-OESTE",
"ES":"SUDESTE","MG":"SUDESTE","RJ":"SUDESTE","SP":"SUDESTE",
"PR":"SUL","RS":"SUL","SC":"SUL"}
APPROVED_CATEGORY_MAPPINGS={}
correction_log=[]
feature_catalog=[]

def strip_accents(text): return "".join(c for c in unicodedata.normalize("NFKD",text) if not unicodedata.combining(c))
def key(value):
    if pd.isna(value): return ""
    text=re.sub(r"\s+"," ",unicodedata.normalize("NFKC",str(value)).strip()).upper()
    return strip_accents(text)
CANONICAL_MISSING={key(x) for x in TEXT_MISSING_TOKENS}
CANONICAL_NEGATIVES={key(k):v for k,v in NEGATIVE_EQUIVALENTS.items()}

def log_change(rule,column,before,after,notes=""):
    a=before.astype("string").fillna("<NA>"); b=after.astype("string").fillna("<NA>")
    changed=a.ne(b)
    if changed.any():
        pairs=pd.DataFrame({"original":a[changed],"corrected":b[changed]}).value_counts().reset_index(name="affected_cells")
        for _,r in pairs.iterrows(): correction_log.append({"timestamp_utc":RUN_TIMESTAMP,"rule":rule,"column":column,"original_value":r["original"],"corrected_value":r["corrected"],"affected_cells":int(r["affected_cells"]),"notes":notes})
    return int(changed.sum())

def add_feature(name,domain,source,definition,feature_type,leakage_risk="low"):
    feature_catalog.append({"feature":name,"domain":domain,"source_columns":source,"definition":definition,"type":feature_type,"leakage_risk":leakage_risk})


## 3.4 Normalize column names and text


In [30]:
def normalize_column(name):
    s=strip_accents(unicodedata.normalize("NFKC",str(name))).lower()
    return re.sub(r"[^0-9a-z]+","_",s).strip("_")
def normalize_text(v):
    if pd.isna(v): return pd.NA
    return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",str(v)).strip()).upper()

if df_clean is not None:
    original_columns=list(df_clean.columns)
    normalized=[normalize_column(c) for c in original_columns]
    if pd.Index(normalized).duplicated().any(): raise ValueError("Column normalization created duplicate names")
    df_clean.columns=normalized
    column_name_map=pd.DataFrame({"original_column":original_columns,"clean_column":normalized})
    text_columns=df_clean.select_dtypes(include=["object","string","category"]).columns
    formatting_summary=[]
    for c in text_columns:
        before=df_clean[c].copy(); after=before.map(normalize_text).astype("string")
        formatting_summary.append({"column":c,"changed_cells":log_change("normalize_text",c,before,after,"NFKC, trim, collapse spaces, uppercase, preserve accents")})
        df_clean[c]=after
    formatting_summary=pd.DataFrame(formatting_summary)


## 3.5 Convert approved textual missing tokens


In [31]:
if df_clean is not None:
    semantic_missing_summary=[]
    for c in df_clean.select_dtypes(include=["object","string","category"]).columns:
        before=df_clean[c].copy(); mask=before.notna() & before.map(key).isin(CANONICAL_MISSING)
        after=before.mask(mask,pd.NA).astype("string")
        changed=log_change("text_missing_to_na",c,before,after,"Numeric sentinel codes excluded by research decision")
        df_clean[c]=after
        semantic_missing_summary.append({"column":c,"converted_to_na":changed})
    semantic_missing_summary=pd.DataFrame(semantic_missing_summary)


## 3.6 Domain-specific standardization before feature engineering

This section applies the research rules supplied for negative classes, age groups, motivation, municipality and country fields, relationship labels, suspect branch labels, constant columns, and the violation hierarchy. Every transformation preserves the original source columns where a derived representation is created.


In [32]:
# 3.6.1 Uniform negative class
NEGATIVE_COLUMNS = [
    "deficiencia_da_vitima", "doenca_rara_da_vitima",
    "deficiencia_do_suspeito", "doenca_rara_do_suspeito",
    "deficiencia_relacionada_a_doenca_rara",
    "deficiencia_relacionada_a_doenca_rara_suspeito",
]
negative_summary=[]
if df_clean is not None:
    for column in [c for c in NEGATIVE_COLUMNS if c in df_clean.columns]:
        before=df_clean[column].copy()
        after=before.map(lambda x: CANONICAL_NEGATIVES.get(key(x), x) if not pd.isna(x) else pd.NA).astype("string")
        changed=log_change("uniform_negative_class",column,before,after,"NÃO TEM DEFICIÊNCIA and NÃO TEM DOENÇA RARA standardized to NÃO")
        df_clean[column]=after
        negative_summary.append({"column":column,"changed_cells":changed})
negative_summary=pd.DataFrame(negative_summary)
display(negative_summary)


""


In [33]:
# 3.6.2 Age-group recoding
MAPA_FAIXA_ETARIA = {
    "VIDA INTRAUTERINA": "Vida intrauterina",
    "RECÉM-NASCIDO (ATÉ 28 DIAS)": "Primeira infância (0 a 6 anos)",
    "RECÉM NASCIDO (ATÉ 90 DIAS ? 3 MESES)": "Primeira infância (0 a 6 anos)",
    "MENOS DE 01 ANO": "Primeira infância (0 a 6 anos)",
    "00 A 01 ANO": "Primeira infância (0 a 6 anos)",
    "01 ANO": "Primeira infância (0 a 6 anos)",
    "02 A 04 ANOS": "Primeira infância (0 a 6 anos)", "02 ANOS": "Primeira infância (0 a 6 anos)",
    "03 ANOS": "Primeira infância (0 a 6 anos)", "04 ANOS": "Primeira infância (0 a 6 anos)",
    "05 A 06 ANOS": "Primeira infância (0 a 6 anos)", "05 ANOS": "Primeira infância (0 a 6 anos)",
    "06 ANOS": "Primeira infância (0 a 6 anos)",
    "07 A 09 ANOS": "Criança (7 a 11 anos)", "07 ANOS": "Criança (7 a 11 anos)",
    "08 ANOS": "Criança (7 a 11 anos)", "09 ANOS": "Criança (7 a 11 anos)",
    "10 A 11 ANOS": "Criança (7 a 11 anos)", "10 ANOS": "Criança (7 a 11 anos)",
    "11 ANOS": "Criança (7 a 11 anos)",
    "12 A 14 ANOS": "Adolescente (12 a 17 anos)", "12 ANOS": "Adolescente (12 a 17 anos)",
    "13 ANOS": "Adolescente (12 a 17 anos)", "14 ANOS": "Adolescente (12 a 17 anos)",
    "15 A 17 ANOS": "Adolescente (12 a 17 anos)", "15 ANOS": "Adolescente (12 a 17 anos)",
    "16 ANOS": "Adolescente (12 a 17 anos)", "17 ANOS": "Adolescente (12 a 17 anos)",
    "CRIANÇA/ADOLESCENTE IDADE NÃO INFORMADA": "Criança ou adolescente (idade exata não informada)",
    "18 A 19 ANOS": "Jovem (18 a 29 anos)", "20 A 24 ANOS": "Jovem (18 a 29 anos)",
    "25 A 29 ANOS": "Jovem (18 a 29 anos)",
    "30 A 34 ANOS": "Adulto (30 a 59 anos)", "35 A 39 ANOS": "Adulto (30 a 59 anos)",
    "40 A 44 ANOS": "Adulto (30 a 59 anos)", "45 A 49 ANOS": "Adulto (30 a 59 anos)",
    "50 A 54 ANOS": "Adulto (30 a 59 anos)", "55 A 59 ANOS": "Adulto (30 a 59 anos)",
    "60 A 64 ANOS": "Pessoa idosa (60 anos ou mais)",
    "65 A 69 ANOS": "Pessoa idosa (60 anos ou mais)",
    "70 A 74 ANOS": "Pessoa idosa (60 anos ou mais)",
    "75 A 79 ANOS": "Pessoa idosa (60 anos ou mais)",
    "80 A 84 ANOS": "Pessoa idosa (60 anos ou mais)",
    "85 A 89 ANOS": "Pessoa idosa (60 anos ou mais)",
    "90 ANOS OU MAIS": "Pessoa idosa (60 anos ou mais)",
    "IDOSO IDADE NÃO INFORMADA": "Pessoa idosa (60 anos ou mais)",
}
ORDEM_FAIXA_ETARIA = {
    "Vida intrauterina":0, "Primeira infância (0 a 6 anos)":1,
    "Criança (7 a 11 anos)":2, "Adolescente (12 a 17 anos)":3,
    "Criança ou adolescente (idade exata não informada)":4,
    "Jovem (18 a 29 anos)":5, "Adulto (30 a 59 anos)":6,
    "Pessoa idosa (60 anos ou mais)":7,
    "Pessoa idosa (idade exata não informada)":8,
}
if df_clean is not None and "faixa_etaria_da_vitima" in df_clean.columns:
    age_source=df_clean["faixa_etaria_da_vitima"].astype("string").str.strip().str.upper()
    df_clean["faixa_etaria_recodificada"]=age_source.map(MAPA_FAIXA_ETARIA).astype("string")
    df_clean["faixa_etaria_ordinal"]=df_clean["faixa_etaria_recodificada"].map(ORDEM_FAIXA_ETARIA).astype("Int64")
    categorias_nao_mapeadas=age_source[age_source.notna() & df_clean["faixa_etaria_recodificada"].isna()].value_counts().rename_axis("categoria_original").reset_index(name="frequencia")
    add_feature("faixa_etaria_recodificada","victim","faixa_etaria_da_vitima","Research age-group recoding","ordered categorical")
    add_feature("faixa_etaria_ordinal","victim","faixa_etaria_recodificada","Ordinal code for recoded age group","integer")
    display(df_clean["faixa_etaria_recodificada"].value_counts(dropna=False))
    display(categorias_nao_mapeadas)
else:
    categorias_nao_mapeadas=pd.DataFrame()


In [34]:
# 3.6.3 Preserve the special gender category as a valid category
SPECIAL_GENDER_CATEGORY = "NÃO SE APLICA - VÍTIMA COMUNIDADE/FAMÍLIA"
if df_clean is not None and "genero_da_vitima" in df_clean.columns:
    df_clean["genero_vitima_comunidade_familia"]=(df_clean["genero_da_vitima"]==SPECIAL_GENDER_CATEGORY).astype("boolean")
    add_feature("genero_vitima_comunidade_familia","victim","genero_da_vitima","Victim represents community/family rather than an individual gender category","boolean")


In [35]:
# 3.6.4 Motivation cleanup
MOTIVATION_PREFIXES = ["AGRAVANTE", "ELEMENTO CIRCUNSTANCIAL", "MOTIVAÇÃO"]
if df_clean is not None and "motivacao" in df_clean.columns:
    before=df_clean["motivacao"].copy()
    cleaned=before.astype("string")
    cleaned=cleaned.str.replace(r"^(AGRAVANTE|ELEMENTO CIRCUNSTANCIAL|MOTIVAÇÃO)\s*[:\-]?\s*","",regex=True)
    exploration_mask=cleaned.str.contains("PARA FINS DE EXPLORAÇÃO DO TRABALHO",na=False)
    cleaned.loc[exploration_mask]=cleaned.loc[exploration_mask].str.split(".",n=1,regex=False).str[0].str.strip()
    df_clean["motivacao_limpa"]=cleaned.astype("string")
    log_change("clean_motivacao","motivacao",before,df_clean["motivacao_limpa"],"Remove prefixes and retain first sentence for labor-exploitation phrase")
    add_feature("motivacao_limpa","complaint","motivacao","Motivation after prefix removal and specified first-chunk rule","categorical")


In [36]:
# 3.6.5 Split municipality code and name
MUNICIPALITY_COLUMNS = [
    "municipio", "municipio_da_vitima", "sl_suspeito_municipio",
    "sl_suspeito_naturalizado_municipio", "sl_vitima_naturalizado_municipio"
]
MUNICIPALITY_UF = {
    "municipio":"uf", "municipio_da_vitima":"uf_da_vitima",
    "sl_suspeito_municipio":"uf_do_suspeito",
    "sl_suspeito_naturalizado_municipio":"sl_suspeito_naturalizado_uf",
    "sl_vitima_naturalizado_municipio":"sl_vitima_naturalizado_uf",
}

def split_municipality(value):
    if pd.isna(value): return pd.NA,pd.NA
    text=str(value).strip()
    parts=[p.strip() for p in text.split("|",maxsplit=1)]
    if len(parts)==2 and re.fullmatch(r"\d{6,7}",parts[0]): return parts[0],parts[1] or pd.NA
    return pd.NA,text or pd.NA

municipality_split_summary=[]
if df_clean is not None:
    for column in [c for c in MUNICIPALITY_COLUMNS if c in df_clean.columns]:
        parsed=df_clean[column].map(split_municipality)
        code_col=f"{column}_codigo_ibge_origem"
        name_col=f"{column}_nome"
        df_clean[code_col]=parsed.map(lambda x:x[0]).astype("string")
        df_clean[name_col]=parsed.map(lambda x:x[1]).astype("string")
        uf_col=MUNICIPALITY_UF.get(column)
        if uf_col in df_clean.columns:
            df_clean[f"{column}_chave_busca_ibge"]=(df_clean[uf_col].astype("string").fillna("")+"|"+df_clean[name_col].astype("string").fillna("")).replace("|",pd.NA).astype("string")
            add_feature(f"{column}_chave_busca_ibge","geographic",f"{uf_col}, {name_col}","Composite UF|municipality key for IBGE lookup","string")
        add_feature(code_col,"geographic",column,"Source IBGE code when present; accepts 6 or 7 digits","string")
        add_feature(name_col,"geographic",column,"Municipality name extracted from source","string")
        municipality_split_summary.append({"column":column,"source_codes_present":int(df_clean[code_col].notna().sum()),"names_present":int(df_clean[name_col].notna().sum())})
municipality_split_summary=pd.DataFrame(municipality_split_summary)
display(municipality_split_summary)


""


In [37]:
# 3.6.6 Split country code and name
COUNTRY_COLUMNS=["nacionalidade_da_vitima","pais","pais_da_vitima","pais_do_suspeito","suspeito_nacionalidade"]
def split_country(value):
    if pd.isna(value): return pd.NA,pd.NA
    text=str(value).strip(); parts=[p.strip() for p in text.split("|",maxsplit=1)]
    if len(parts)==2 and re.fullmatch(r"[A-Z]{2,3}",key(parts[0])): return key(parts[0]),parts[1] or pd.NA
    return pd.NA,text or pd.NA
country_split_summary=[]
if df_clean is not None:
    for column in [c for c in COUNTRY_COLUMNS if c in df_clean.columns]:
        parsed=df_clean[column].map(split_country)
        code_col=f"{column}_codigo_pais"
        name_col=f"{column}_nome_pais"
        df_clean[code_col]=parsed.map(lambda x:x[0]).astype("string")
        df_clean[name_col]=parsed.map(lambda x:x[1]).astype("string")
        add_feature(code_col,"geographic",column,"Country code extracted before |; missing when absent","string")
        add_feature(name_col,"geographic",column,"Country name extracted from source","string")
        country_split_summary.append({"column":column,"codes_present":int(df_clean[code_col].notna().sum()),"names_present":int(df_clean[name_col].notna().sum())})
country_split_summary=pd.DataFrame(country_split_summary)
display(country_split_summary)


""


In [38]:
# 3.6.7 Relationship and suspect-branch formatting
if df_clean is not None and "relacao_vitima_suspeito" in df_clean.columns:
    before=df_clean["relacao_vitima_suspeito"].copy()
    after=before.str.replace(r"\s+\(A\)","(A)",regex=True).astype("string")
    log_change("remove_space_before_parenthetical_a","relacao_vitima_suspeito",before,after,"AMIGO (A) -> AMIGO(A)")
    df_clean["relacao_vitima_suspeito"]=after
if df_clean is not None and "sl_suspeito_ramo" in df_clean.columns:
    before=df_clean["sl_suspeito_ramo"].copy()
    after=before.str.replace("?","-",regex=False).astype("string")
    log_change("replace_question_with_hyphen","sl_suspeito_ramo",before,after)
    df_clean["sl_suspeito_ramo"]=after


In [39]:
# 3.6.8 Remove constant variable after documenting it
removed_constant_columns=[]
if df_clean is not None and "sl_vitima_naturalidade" in df_clean.columns:
    unique_non_null=df_clean["sl_vitima_naturalidade"].nunique(dropna=True)
    if unique_non_null<=1:
        removed_constant_columns.append({"column":"sl_vitima_naturalidade","unique_non_null":int(unique_non_null),"reason":"constant or empty"})
        df_clean=df_clean.drop(columns=["sl_vitima_naturalidade"])
removed_constant_columns=pd.DataFrame(removed_constant_columns)
display(removed_constant_columns)


""


In [40]:
# 3.6.9 Expand violation hierarchy into four levels
VIOLATION_LEVEL_LABELS = {
    1:"Direitos a serem tutelados sob a perspectiva dos direitos humanos",
    2:"Bem jurídico em sentido estrito",
    3:"Espécies de violações a determinado bem jurídico protegido",
    4:"Detalhamento de violações a determinada espécie",
}
if df_clean is not None and "violacao" in df_clean.columns:
    parts=df_clean["violacao"].astype("string").str.split(">",expand=True)
    for level in range(1,5):
        col=f"violacao_nivel_{level}"
        df_clean[col]=(parts[level-1].str.strip() if level-1 in parts.columns else pd.Series(pd.NA,index=df_clean.index,dtype="string")).astype("string")
        add_feature(col,"violation","violacao",VIOLATION_LEVEL_LABELS[level],"categorical")
    df_clean["violacao_nivel_3_originalmente_ausente"]=df_clean["violacao_nivel_3"].isna().astype("boolean")
    df_clean["violacao_nivel_3"]=df_clean["violacao_nivel_3"].fillna(df_clean["violacao_nivel_2"]).fillna(df_clean["violacao_nivel_1"])
    df_clean["violacao_numero_niveis_informados"]=parts.notna().sum(axis=1).clip(upper=4).astype("Int64")
    df_clean["violacao_numero_niveis_gerados"]=pd.Series(4,index=df_clean.index,dtype="Int64").where(df_clean["violacao"].notna())
    add_feature("violacao_nivel_3_originalmente_ausente","violation","violacao_nivel_3","Flags level-3 backfill","boolean")
    add_feature("violacao_numero_niveis_informados","violation","violacao","Number of source hierarchy chunks, capped at four","integer")
    add_feature("violacao_numero_niveis_gerados","violation","violacao","Number of generated hierarchy columns","integer")


## 3.6 Apply approved mappings and data types


In [41]:
if df_clean is not None:
    mapping_summary=[]
    for c,mapping in APPROVED_CATEGORY_MAPPINGS.items():
        if c not in df_clean: continue
        canonical={key(k):v for k,v in mapping.items()}
        before=df_clean[c].copy(); after=before.map(lambda x:canonical.get(key(x),x) if not pd.isna(x) else pd.NA).astype("string")
        changed=log_change("approved_mapping",c,before,after)
        df_clean[c]=after; mapping_summary.append({"column":c,"changed_cells":changed})
    mapping_summary=pd.DataFrame(mapping_summary)
    if "data_de_cadastro" in df_clean: df_clean["data_de_cadastro"]=pd.to_datetime(df_clean["data_de_cadastro"],errors="coerce")
    if "sl_quantidade_vitimas" in df_clean:
        q=pd.to_numeric(df_clean["sl_quantidade_vitimas"],errors="coerce")
        if not (q.dropna()%1!=0).any(): df_clean["sl_quantidade_vitimas"]=q.astype("Int64")


# Part B: Feature Engineering

Derived features are organized by analytical domain. Raw columns remain available so every transformation can be audited or revised.


## 3.7 Temporal features

Features include calendar fields, weekend status, time-of-day bands, quarter, semester, ISO week, and elapsed time from the first observed registration. These support Steps 6 and 9 without repeatedly transforming dates.


In [42]:
if df_clean is not None and "data_de_cadastro" in df_clean:
    d=df_clean["data_de_cadastro"]
    temporal={
        "ano_cadastro":d.dt.year.astype("Int64"),
        "trimestre_cadastro":d.dt.quarter.astype("Int64"),
        "semestre_cadastro":pd.Series(np.where(d.dt.month<=6,1,2),index=df_clean.index).where(d.notna()).astype("Int64"),
        "mes_cadastro":d.dt.month.astype("Int64"),
        "ano_mes_cadastro":d.dt.to_period("M").astype("string"),
        "semana_iso_cadastro":d.dt.isocalendar().week.astype("Int64"),
        "dia_mes_cadastro":d.dt.day.astype("Int64"),
        "dia_semana_num":d.dt.dayofweek.astype("Int64"),
        "dia_semana_cadastro":d.dt.day_name().astype("string"),
        "fim_de_semana":d.dt.dayofweek.isin([5,6]).astype("boolean"),
        "hora_cadastro":d.dt.hour.astype("Int64"),
        "dias_desde_primeiro_registro":(d-d.min()).dt.days.astype("Int64"),
    }
    for name,series in temporal.items():
        df_clean[name]=series; add_feature(name,"temporal","data_de_cadastro",name.replace("_"," "),str(series.dtype))
    hour=df_clean["hora_cadastro"]
    df_clean["periodo_dia_cadastro"]=pd.cut(hour,bins=[-1,5,11,17,23],labels=["MADRUGADA","MANHÃ","TARDE","NOITE"]).astype("string")
    add_feature("periodo_dia_cadastro","temporal","data_de_cadastro","Time-of-day band from registration hour","categorical")


## 3.8 Geographic features

The notebook creates validated UF keys, Brazilian macro-regions, domestic versus international indicators, and location-completeness scores. Municipality codes should be added later through an official IBGE reference join, not inferred from names.


In [43]:
if df_clean is not None:
    geo_specs=[("pais","uf","municipio","denuncia"),("pais_da_vitima","uf_da_vitima","municipio_da_vitima","vitima"),("pais_do_suspeito","uf_do_suspeito","sl_suspeito_municipio","suspeito")]
    for country,uf,municipality,prefix in geo_specs:
        if uf in df_clean:
            df_clean[f"uf_{prefix}_valida"]=df_clean[uf].isin(UF_CODES).astype("boolean")
            df_clean[f"regiao_{prefix}"]=df_clean[uf].map(REGION_BY_UF).astype("string")
            add_feature(f"uf_{prefix}_valida","geographic",uf,"UF belongs to 27 official codes","boolean")
            add_feature(f"regiao_{prefix}","geographic",uf,"Brazilian macro-region derived from UF","categorical")
        if country in df_clean:
            df_clean[f"localizacao_{prefix}_brasil"]=df_clean[country].map(key).isin({"BRASIL","BRAZIL","BR"}).astype("boolean")
            add_feature(f"localizacao_{prefix}_brasil","geographic",country,"Country indicates Brazil","boolean")
        available=[c for c in [country,uf,municipality] if c in df_clean]
        if available:
            df_clean[f"completude_localizacao_{prefix}"]=df_clean[available].notna().sum(axis=1).astype("Int64")
            add_feature(f"completude_localizacao_{prefix}","geographic",", ".join(available),"Count of populated country, UF, municipality fields","integer")


## 3.9 Victim profile features

These features summarize whether key victim attributes are present and create broad, auditable indicators for vulnerability, disability, rare disease, incarceration, and demographic-profile completeness. They do not infer characteristics absent from the source.


In [44]:
def yes_indicator(series):
    return series.map(key).isin({"SIM","S","YES","1","TRUE"}).astype("boolean")

if df_clean is not None:
    victim_profile=[c for c in ["genero_da_vitima","orientacao_sexual_da_vitima","faixa_etaria_da_vitima","nacionalidade_da_vitima","deficiencia_da_vitima","doenca_rara_da_vitima","vitima_preso_a","profissao_da_vitima","grau_de_instrucao_da_vitima","religiao_da_vitima","raca_cor_da_vitima","etnia_da_vitima","faixa_de_renda_da_vitima"] if c in df_clean]
    if victim_profile:
        df_clean["campos_perfil_vitima_preenchidos"]=df_clean[victim_profile].notna().sum(axis=1).astype("Int64")
        df_clean["pct_perfil_vitima_preenchido"]=(df_clean["campos_perfil_vitima_preenchidos"]/len(victim_profile)*100).round(2)
        add_feature("campos_perfil_vitima_preenchidos","victim",", ".join(victim_profile),"Number of populated victim profile fields","integer")
        add_feature("pct_perfil_vitima_preenchido","victim",", ".join(victim_profile),"Percentage of populated victim profile fields","numeric")
    for source,target in [("deficiencia_da_vitima","vitima_com_deficiencia"),("doenca_rara_da_vitima","vitima_com_doenca_rara"),("vitima_preso_a","vitima_em_privacao_liberdade")]:
        if source in df_clean:
            df_clean[target]=yes_indicator(df_clean[source]); add_feature(target,"victim",source,"Positive-response indicator","boolean")
    if "faixa_etaria_da_vitima" in df_clean:
        agekey=df_clean["faixa_etaria_da_vitima"].map(key)
        df_clean["vitima_crianca_adolescente"]=agekey.str.contains(r"CRIAN|ADOLESC|0 A|1[0-7]",regex=True,na=False).astype("boolean")
        df_clean["vitima_idosa"]=agekey.str.contains(r"IDOS|60|65|70|80",regex=True,na=False).astype("boolean")
        add_feature("vitima_crianca_adolescente","victim","faixa_etaria_da_vitima","Broad child/adolescent label indicator; review against official age domain","boolean")
        add_feature("vitima_idosa","victim","faixa_etaria_da_vitima","Broad older-person label indicator; review against official age domain","boolean")


## 3.10 Suspect profile features


In [45]:
if df_clean is not None:
    suspect_profile=[c for c in ["natureza_juridica_do_suspeito","genero_do_suspeito","orientacao_sexual_do_suspeito","faixa_etaria_do_suspeito","suspeito_nacionalidade","deficiencia_do_suspeito","doenca_rara_do_suspeito","suspeito_preso","profissao_do_suspeito","grau_de_instrucao_do_suspeito","religiao_do_suspeito","raca_cor_do_suspeito","suspeito_etnia","faixa_de_renda_do_suspeito"] if c in df_clean]
    if suspect_profile:
        df_clean["campos_perfil_suspeito_preenchidos"]=df_clean[suspect_profile].notna().sum(axis=1).astype("Int64")
        df_clean["pct_perfil_suspeito_preenchido"]=(df_clean["campos_perfil_suspeito_preenchidos"]/len(suspect_profile)*100).round(2)
        add_feature("campos_perfil_suspeito_preenchidos","suspect",", ".join(suspect_profile),"Number of populated suspect profile fields","integer")
        add_feature("pct_perfil_suspeito_preenchido","suspect",", ".join(suspect_profile),"Percentage of populated suspect profile fields","numeric")
    for source,target in [("deficiencia_do_suspeito","suspeito_com_deficiencia"),("doenca_rara_do_suspeito","suspeito_com_doenca_rara"),("suspeito_preso","suspeito_em_privacao_liberdade")]:
        if source in df_clean:
            df_clean[target]=yes_indicator(df_clean[source]); add_feature(target,"suspect",source,"Positive-response indicator","boolean")
    if "natureza_juridica_do_suspeito" in df_clean:
        k=df_clean["natureza_juridica_do_suspeito"].map(key)
        df_clean["suspeito_pessoa_juridica"]=k.str.contains(r"JURIDICA|ORGAO|EMPRESA|INSTITU",regex=True,na=False).astype("boolean")
        add_feature("suspeito_pessoa_juridica","suspect","natureza_juridica_do_suspeito","Legal-entity or institution label indicator","boolean")


## 3.11 Relationship and complaint-context features

Relationship categories are grouped into broad analytical contexts only through transparent label rules. These groupings must be compared with the official data dictionary before publication.


In [46]:
if df_clean is not None:
    if "relacao_vitima_suspeito" in df_clean:
        r=df_clean["relacao_vitima_suspeito"].map(key)
        conditions=[
            r.str.contains(r"PAI|MAE|MÃE|PADRAST|MADRAST|FILH|IRMA|IRMÃ|AVO|AVÓ|TIO|TIA|PARENT",regex=True,na=False),
            r.str.contains(r"CONJUGE|CÔNJUGE|COMPANHEIR|NAMORAD|EX-|MARIDO|ESPOS",regex=True,na=False),
            r.str.contains(r"CUIDADOR|PROFISSIONAL|SERVIDOR|EMPREGADOR|INSTITU",regex=True,na=False),
            r.str.contains(r"VIZINH|CONHECID|AMIG",regex=True,na=False),
            r.str.contains(r"DESCONHECID|SEM RELACAO|SEM RELAÇÃO",regex=True,na=False),
        ]
        choices=["FAMILIAR","PARCEIRO_INTIMO","INSTITUCIONAL_PROFISSIONAL","COMUNITARIO_CONHECIDO","DESCONHECIDO"]
        df_clean["contexto_relacao_vitima_suspeito"]=pd.Series(np.select(conditions,choices,default="OUTRO"),index=df_clean.index,dtype="string").where(df_clean["relacao_vitima_suspeito"].notna(),pd.NA)
        add_feature("contexto_relacao_vitima_suspeito","relationship","relacao_vitima_suspeito","Broad relationship context from transparent label rules","categorical")
    if "denuncia_emergencial" in df_clean:
        df_clean["denuncia_emergencial_flag"]=yes_indicator(df_clean["denuncia_emergencial"])
        add_feature("denuncia_emergencial_flag","complaint","denuncia_emergencial","Positive emergency-response indicator","boolean",leakage_risk="depends_on_target")
    if "sl_quantidade_vitimas" in df_clean:
        q=pd.to_numeric(df_clean["sl_quantidade_vitimas"],errors="coerce")
        df_clean["denuncia_coletiva"]=(q>1).astype("boolean").where(q.notna())
        df_clean["faixa_quantidade_vitimas"]=pd.cut(q,bins=[0,1,2,5,10,np.inf],labels=["1","2","3-5","6-10","11+"]).astype("string")
        df_clean["log1p_quantidade_vitimas"]=np.log1p(q.astype(float))
        add_feature("denuncia_coletiva","complaint","sl_quantidade_vitimas","More than one victim","boolean")
        add_feature("faixa_quantidade_vitimas","complaint","sl_quantidade_vitimas","Victim-count band","ordinal categorical")
        add_feature("log1p_quantidade_vitimas","complaint","sl_quantidade_vitimas","Natural log of 1 plus victim count","numeric")


## 3.12 Violation features

The raw `violacao` field remains authoritative. The notebook adds text-depth and optional hierarchy fields. Official violation families should be supplied through `VIOLATION_FAMILY_MAP` rather than inferred silently.


In [47]:
VIOLATION_FAMILY_MAP={
    # "VIOLÊNCIA FÍSICA": "INTEGRIDADE_FISICA",
    # Populate with official codebook categories.
}
if df_clean is not None and "violacao" in df_clean:
    v=df_clean["violacao"].astype("string")
    df_clean["violacao_nivel_profundidade"]=(v.str.count(r"[>/|;]")+1).where(v.notna()).astype("Int64")
    df_clean["violacao_multirrotulo_textual"]=v.str.contains(r"[|;]",regex=True,na=False).astype("boolean").where(v.notna())
    canonical_map={key(k):value for k,value in VIOLATION_FAMILY_MAP.items()}
    df_clean["familia_violacao"]=v.map(lambda x:canonical_map.get(key(x),pd.NA) if not pd.isna(x) else pd.NA).astype("string")
    add_feature("violacao_nivel_profundidade","violation","violacao","Number of textual hierarchy levels separated by >, /, |, or ;","integer")
    add_feature("violacao_multirrotulo_textual","violation","violacao","Text contains multi-label delimiter | or ;","boolean")
    add_feature("familia_violacao","violation","violacao","Official violation family mapping; empty until map supplied","categorical")


## 3.13 Record completeness and quality features

These features support sensitivity analysis and detection of documentation-driven clusters. They should not be used as substantive victim or suspect characteristics.


In [48]:
if df_clean is not None:
    source_columns=[c for c in original_columns if normalize_column(c) in df_clean.columns]
    clean_source_names=[normalize_column(c) for c in source_columns]
    df_clean["campos_fonte_preenchidos"]=df_clean[clean_source_names].notna().sum(axis=1).astype("Int64")
    df_clean["pct_campos_fonte_preenchidos"]=(df_clean["campos_fonte_preenchidos"]/len(clean_source_names)*100).round(2)
    df_clean["registro_baixa_completude"]=(df_clean["pct_campos_fonte_preenchidos"]<50).astype("boolean")
    add_feature("campos_fonte_preenchidos","quality",", ".join(clean_source_names),"Number of non-missing original source fields","integer")
    add_feature("pct_campos_fonte_preenchidos","quality",", ".join(clean_source_names),"Percentage of non-missing original source fields","numeric")
    add_feature("registro_baixa_completude","quality","pct_campos_fonte_preenchidos","Source-field completeness below 50%","boolean")
    if "hash" in df_clean:
        df_clean["linhas_por_hash"]=df_clean.groupby("hash")["hash"].transform("size").astype("Int64")
        df_clean["hash_repetido"]=(df_clean["linhas_por_hash"]>1).astype("boolean")
        add_feature("linhas_por_hash","quality","hash","Number of rows associated with hash","integer")
        add_feature("hash_repetido","quality","hash","Hash occurs in more than one row","boolean")


## 3.14 Optional modeling matrix

This section is disabled by default. Encoding must occur after train/test partitioning in predictive work to prevent leakage. The exported analytical dataset should remain human-readable.


In [49]:
model_matrix=pd.DataFrame()
if CREATE_MODEL_MATRIX and df_clean is not None:
    excluded={"hash","violacao","familia_violacao"}
    candidate=[c for c in df_clean.columns if c not in excluded and c!=DATE_COLUMN]
    low_cardinality=[c for c in candidate if str(df_clean[c].dtype) in {"string","object","category"} and df_clean[c].nunique(dropna=True)<=30]
    numeric=[c for c in candidate if pd.api.types.is_numeric_dtype(df_clean[c]) or str(df_clean[c].dtype) in {"boolean","bool"}]
    model_matrix=pd.concat([df_clean[numeric],pd.get_dummies(df_clean[low_cardinality],dummy_na=True,dtype="int8")],axis=1)
    print("Model matrix shape:",model_matrix.shape)


## 3.15 Validate transformations and engineered features


In [50]:
if df_clean is not None:
    validation=[]
    validation.append({"check":"row_count_preserved","status":"PASS" if len(df_clean)==len(df_raw) else "REVIEW","value":len(df_clean)})
    validation.append({"check":"column_names_unique","status":"PASS" if df_clean.columns.is_unique else "FAIL","value":df_clean.columns.duplicated().sum()})
    validation.append({"check":"feature_names_unique","status":"PASS" if pd.DataFrame(feature_catalog)["feature"].is_unique else "FAIL","value":len(feature_catalog)})
    if "sl_quantidade_vitimas" in df_clean:
        invalid=(df_clean["sl_quantidade_vitimas"].notna() & (df_clean["sl_quantidade_vitimas"]<=0)).sum()
        validation.append({"check":"victim_quantity_positive","status":"PASS" if invalid==0 else "REVIEW","value":int(invalid)})
    for c in [x for x in df_clean.columns if x.startswith("pct_")]:
        invalid=(df_clean[c].notna() & ~df_clean[c].between(0,100)).sum()
        validation.append({"check":f"{c}_range_0_100","status":"PASS" if invalid==0 else "FAIL","value":int(invalid)})
    transformation_validation=pd.DataFrame(validation)
    feature_catalog_df=pd.DataFrame(feature_catalog)
    display(transformation_validation)
    display(feature_catalog_df)


## 3.16 Before-versus-after summary and feature-domain chart


In [51]:
if df_clean is not None:
    before_after_summary=pd.DataFrame({
        "metric":["Rows","Columns","Missing cells","Memory MB"],
        "raw":[len(df_raw),df_raw.shape[1],int(df_raw.isna().sum().sum()),round(df_raw.memory_usage(deep=True).sum()/1024**2,2)],
        "analysis_ready":[len(df_clean),df_clean.shape[1],int(df_clean.isna().sum().sum()),round(df_clean.memory_usage(deep=True).sum()/1024**2,2)]
    })
    display(before_after_summary)
    domain_counts=feature_catalog_df["domain"].value_counts().rename_axis("domain").reset_index(name="features")
    fig=px.bar(domain_counts,x="domain",y="features",text="features",title="Engineered features by analytical domain")
    fig.update_layout(template="plotly_white")
    fig.show()


## 3.17 Export the analysis-ready layer and audit artifacts


In [52]:
if df_clean is not None:
    correction_log_df=pd.DataFrame(correction_log)
    cleaning_summary=(correction_log_df.groupby(["rule","column"],as_index=False).agg(affected_cells=("affected_cells","sum"),distinct_original_values=("original_value","nunique")) if not correction_log_df.empty else pd.DataFrame())
    artifacts={
        "column_name_map.csv":column_name_map,
        "formatting_summary.csv":formatting_summary,
        "semantic_missing_summary.csv":semantic_missing_summary,
        "mapping_summary.csv":mapping_summary,
        "correction_log.csv":correction_log_df,
        "cleaning_summary.csv":cleaning_summary,
        "feature_catalog.csv":feature_catalog_df,
        "transformation_validation.csv":transformation_validation,
        "before_after_summary.csv":before_after_summary,
        "age_categories_unmapped.csv":categorias_nao_mapeadas,
        "municipality_split_summary.csv":municipality_split_summary,
        "country_split_summary.csv":country_split_summary,
        "removed_constant_columns.csv":removed_constant_columns,
    }
    for filename,table in artifacts.items(): table.to_csv(OUTPUT_DIR/filename,index=False,encoding="utf-8-sig")
    if CREATE_MODEL_MATRIX: model_matrix.to_parquet(OUTPUT_DIR/"RESTRICTED_model_matrix.parquet",index=False)
    if EXPORT_CLEAN_DATA:
        if EXPORT_FORMAT=="parquet":
            try: df_clean.to_parquet(OUTPUT_DIR/"disque100_clean.parquet",index=False); exported=OUTPUT_DIR/"disque100_clean.parquet"
            except ImportError: df_clean.to_csv(OUTPUT_DIR/"disque100_clean.csv",index=False,encoding="utf-8-sig"); exported=OUTPUT_DIR/"disque100_clean.csv"
        else:
            df_clean.to_csv(OUTPUT_DIR/"disque100_clean.csv",index=False,encoding="utf-8-sig"); exported=OUTPUT_DIR/"disque100_clean.csv"
    metadata={"run_timestamp_utc":RUN_TIMESTAMP,"source_path":str(DATA_PATH),"source_sha256":source_sha256,"source_rows":len(df_raw),"source_columns":df_raw.shape[1],"analysis_rows":len(df_clean),"analysis_columns":df_clean.shape[1],"engineered_features":len(feature_catalog_df),"numeric_missing_codes_processed":False,"exact_duplicates_removed":REMOVE_EXACT_DUPLICATES}
    (OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Exported:",exported.resolve())


## 3.18 Feature review checklist

Before Steps 4 through 9:

1. Validate age-related label rules against the official age-group domain.
2. Validate broad relationship contexts with domain specialists.
3. Populate `VIOLATION_FAMILY_MAP` from the official violation hierarchy.
4. Join official IBGE municipality codes rather than relying on names.
5. Review whether emergency status is an outcome, predictor, or stratifier before modeling.
6. Keep quality/completeness features separate from substantive profile features.
7. Confirm that repeated hashes represent legitimate one-to-many records.
8. Check temporal features for date-only timestamps before using hour or period of day.
9. Fit encoders only within training data in predictive workflows.
10. Document every engineered feature used in a paper or model from `feature_catalog.csv`.

## Next step

Step 4 should read `disque100_clean.parquet`, which now contains both cleaned source variables and engineered analytical features. The later notebooks may be expanded to use `regiao_denuncia`, `contexto_relacao_vitima_suspeito`, profile-completeness measures, victim-count bands, and other reviewed features.
